# M2 · Tool calling: model wywołuje Twoje funkcje

> VP of Sales, TechRetail Corp: *"ai_query() jest fajne, ale to JA muszę napisać SQL i podać dane. A gdyby model SAM mógł sięgnąć po dane, których potrzebuje?"*

W M1 model nie znał danych TechRetail. Teraz odwracamy kierunek: to model decyduje, kiedy wywołać Twoją funkcję i z jakimi parametrami. To fundament każdego agenta.

| Część | Co powstaje | Lab |
|---|---|---|
| 1 | funkcja `get_revenue_summary` w Unity Catalog | opis (COMMENT) i warunek WHERE |
| 2 | tool calling "ręcznie" w Pythonie: model wybiera funkcję i parametry | schemat `tools` (opcjonalnie) |
| 3 | trzy narzędzia agenta: średnia wartość, profil bez PII, formatowanie | brak (gotowy kod, z tych funkcji korzystają M5 i M6) |
| 4 | test surowym payloadem, czyli co dokładnie zobaczy model | brak |
| 5 | Playground z funkcjami jako Tools: trafienie, PII, fallback | UI |

**Wymaga:** tabeli `gold_customer_360` z `00_setup`. Ścieżka B korzysta z kopii Bakehouse w `workspace.bakehouse`, ścieżka C z tabeli `workspace.airbnb.listings`; obie zakłada `00_setup`.

## Mapa ścieżek

| Ścieżka | Co robisz | Gotowe, gdy | Gdzie |
|---|---|---|---|
| **A · Razem** (TechRetail) | razem z prowadzącym: `get_revenue_summary` (TODO: `COMMENT` i `WHERE`), tool calling "ręcznie", trzy narzędzia agenta, test payloadem, Playground | trzy funkcje agenta leżą w `workspace.default`, test payloadem przechodzi bez `tax_id` | sekcje 1-5 |
| **B · Samodzielnie** (Bakehouse) | dwie funkcje w `workspace.bakehouse` i test, którą z nich wybierze model | obie funkcje zwracają tekst dla istniejącej franczyzy i komunikat dla nieistniejącej, model trafia 2 z 2 | "B · Samodzielnie: dwie funkcje i wybór narzędzia" |
| **C · Wyzwanie** (Airbnb) | funkcja w `workspace.airbnb`, która przy braku danych mówi "brak ofert", i grant, który znika po `CREATE OR REPLACE` | Mission: 789 ofert, zmyślona dzielnica: "Brak ofert...", po `CREATE OR REPLACE` 0 grantów na funkcji dla grupy | "C · Wyzwanie: funkcja, która mówi »nie mam danych«..." |

Ścieżka A to pełny cel modułu. B i C robisz, gdy skończysz A.

In [ ]:
%pip install --quiet -r ../requirements.txt

In [ ]:
dbutils.library.restartPython()

**Infrastruktura.** Konfiguracja wspólna dla wszystkich modułów. Uruchom i czytaj dalej, tu nie ma nic do nauczenia.


In [ ]:
# Wspólna konfiguracja warsztatu. Ta sama komórka jest w każdym notebooku.
CATALOG = "workspace"
SCHEMA = "default"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_360"
VOLUME = "retail_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DOCS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_chunks"
BH_SCHEMA = "bakehouse"       # ścieżka B: kopie danych Bakehouse i funkcje-narzędzia
AIRBNB_SCHEMA = "airbnb"      # ścieżka C: oferty Airbnb i funkcje-narzędzia
POLICY_SCHEMA = "governance"  # maski i filtry ścieżek B i C: poza schematami, które MCP wystawia agentowi
BH_TRANSACTIONS = f"{CATALOG}.{BH_SCHEMA}.transactions"
BH_REVIEWS = f"{CATALOG}.{BH_SCHEMA}.reviews"
AIRBNB_TABLE = f"{CATALOG}.{AIRBNB_SCHEMA}.listings"
SEARCH_ENDPOINT = "retail_rag_search"
SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.retail_rag_chunks_index"
AVG_VALUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_average_customer_value"
PROFILE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_customer_profile"
FORMAT_FUNCTION = f"{CATALOG}.{SCHEMA}.format_customer_for_agent"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
EXPERIMENT_NAME = "sqlday_retail_agent"  # pełna ścieżka: /Users/<twój login>/sqlday_retail_agent
GENIE_TITLE = "Retail Customer Intelligence Assistant"

import logging
# MLflow w notebooku serverless (UI) wypisuje przy tracingu stos Py4JSecurityException z "resolving tags".
# To ostrzeżenie, nie błąd. Trace zapisuje się poprawnie, a wyciszamy je, żeby nikt nie wziął go za błąd.
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

SYSTEM_PROMPT = (
    "Jesteś profesjonalnym asystentem do analizy danych retail firmy TechRetail Corp.\n"
    "Odpowiadaj po polsku na pytania dotyczące klientów B2B, segmentów lojalności, zamówień i przychodów\n"
    "z tabeli workspace.default.gold_customer_360 oraz raportów analityków.\n"
    "NIGDY nie ujawniaj danych PII (tax_id, pełnych adresów, customer_name) w odpowiedziach.\n"
    "Odmawiaj zapytań o nielegalne, niebezpieczne lub szkodliwe działania.\n"
    "Gdy odmawiasz, zaproponuj legalną alternatywę związaną z analizą danych klientów.\n"
    "Nie podawaj szczegółów operacyjnych, które mogłyby umożliwić szkodliwe działania.\n"
    "Liczby podawaj wyłącznie z wyników narzędzi; niczego nie zgaduj.\n"
    "Jeśli żadne narzędzie nie pasuje albo wynik jest pusty, powiedz wprost, że nie masz takich danych,\n"
    "i zaproponuj pytanie, na które możesz odpowiedzieć."
)

**Infrastruktura.** Komórka przygotowuje moduł. Uruchom ją i czytaj dalej.

Tworzy połączenie z API Databricks (`w`) i klienta modelu (`llm`, ten sam co w M1). Wybiera też klienta X, czyli pierwszego klienta VIP według numeru `customer_id`. Jego numer zobaczysz w wydruku. Użyjesz go w teście funkcji, w Playground i w M5.


In [ ]:
import json
import time

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
llm = w.serving_endpoints.get_open_ai_client()
REVENUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_revenue_summary"

# Klient X z macierzy tras (M5): pierwszy klient VIP według customer_id.
VIP_FILTER = "loyalty_segment = 3 AND num_orders > 0 AND city IS NOT NULL AND tax_id IS NOT NULL"
vip_customer = spark.table(GOLD_TABLE).where(VIP_FILTER).orderBy("customer_id").first()
VIP_CUSTOMER_ID = int(vip_customer["customer_id"])

print(f"Tabela: {GOLD_TABLE} | klient X (VIP): {VIP_CUSTOMER_ID}")


## Czym jest narzędzie dla modelu

> **Cel:** zrozumieć, co model naprawdę widzi, gdy dostaje narzędzie.
> **Gotowe, gdy:** potrafisz wymienić cztery rzeczy, które model wie o funkcji, i zauważyć, że nie ma wśród nich kodu.


Model nie widzi kodu SQL ani danych. Widzi tylko cztery rzeczy:

| Element | Przykład | Skąd go bierze Unity Catalog |
|---|---|---|
| **Nazwa** | `get_revenue_summary` | nazwa funkcji |
| **Opis**: do czego i kiedy użyć | "Przychód per segment i stan. Używaj do pytań o przychody." | `COMMENT` funkcji |
| **Parametry** z typami i opisami | `segment` 0-3 albo -1, `state_filter` kod stanu albo `ALL` | `COMMENT` parametrów |
| **Wynik** | tekst albo liczba | `RETURNS` |

**Jak model wybiera.** Pytanie trafia do modelu, model wybiera narzędzie na podstawie opisu, funkcja wykonuje SQL deterministycznie, a model formatuje wynik. Gdy model nie sięga po funkcję albo źle ustawia parametr, poprawiasz opis, nie kod.

**Pięć zasad projektowania narzędzi:**
1. **Małe.** Jedno pytanie biznesowe, jedna funkcja: `get_customer_profile(id)`, a nie `get_everything(...)`.
2. **Jednoznaczne.** Opis mówi, kiedy użyć i kiedy nie. Między dwoma narzędziami o podobnym opisie model wybiera losowo.
3. **Deterministyczne.** Ten sam input daje ten sam output. Żadnego modelu w środku narzędzia.
4. **Bez PII w wyniku.** Agent nie ma czego ujawnić, nawet gdy prompt zawiedzie.
5. **Podział ról.** SQL daje dostęp do danych, Python robi logikę i formatowanie. Funkcja Python w Unity Catalog nie czyta tabel, więc ten podział jest wymuszony.

## 1. Pierwsze narzędzie: `get_revenue_summary`

> **Cel:** pierwsza własna funkcja, którą model może wywołać.
> **Gotowe, gdy:** funkcja jest w katalogu i zwraca wynik na teście payloadem.


`CREATE FUNCTION ... COMMENT '...'` zapisuje funkcję w katalogu obok tabel, pod adresem `workspace.default.get_revenue_summary`. Oba `COMMENT` (funkcji i parametrów) to dokładnie ten opis, który zobaczy model.

Jak działa ta funkcja:

- Ma dwa parametry. `segment` to numer segmentu lojalności (`-1` oznacza wszystkie), a `state_filter` to skrót stanu, np. `NY` (`ALL` oznacza wszystkie).
- Liczy na tabeli Gold liczbę klientów, przychód, średnią na klienta i liczbę zamówień, a potem skleja to w jeden tekst. Ten tekst model dostanie jako wynik narzędzia.
- Gdy żaden wiersz nie pasuje, zwraca zdanie "No data for ...". Bez tego model dostałby pusty wynik i mógłby zgadywać.
- Ostatnia linijka komórki wywołuje funkcję dla VIP-ów w Nowym Jorku, czyli sprawdza ją bez modelu.

**Lab:** uzupełnij opis funkcji tak, żeby model wiedział, kiedy jej użyć. Dopisz też warunek `WHERE`, który obsługuje `-1` (wszystkie segmenty) i `ALL` (wszystkie stany).

In [ ]:
%sql
-- ZADANIE 3: opis narzędzia i warunek filtra.
--
-- Co masz zrobić:
--   1. Napisać COMMENT funkcji. To nie jest dokumentacja dla ludzi, tylko jedyne, co model
--      przeczyta, decydując, czy sięgnąć po tę funkcję.
--   2. Uzupełnić WHERE tak, żeby obsłużył wartości "wszystkie" w obu parametrach.
--
-- Gotowe, gdy: test na dole komórki zwraca raport z liczbami dla VIP w NY,
--              a nie "No data for segment 3 / state NY.".
--
-- Utknąłeś? Rozwiązanie: ../demo/m2_tool_calling, komórka m2-revenue-function
CREATE OR REPLACE FUNCTION workspace.default.get_revenue_summary(
  segment BIGINT COMMENT 'Loyalty segment ID: 0=new/inactive, 1=occasional, 2=regular, 3=VIP. Pass -1 for all segments.',
  state_filter STRING COMMENT 'US state abbreviation (e.g. NY, TX) or ALL for all states. US states only.'
)
RETURNS STRING
-- TODO 1: COMMENT funkcji, po angielsku, 2-3 zdania. Odpowiedz w nim na trzy pytania:
--           CO zwraca      -> jakie liczby i z której tabeli
--           KIEDY użyć     -> "Use to answer business questions about ..."
--           KIEDY NIE użyć -> dwa ograniczenia, które ta funkcja ma naprawdę:
--                             dane obejmują wyłącznie stany USA (brak np. Kanady),
--                             i funkcja nie odpowiada na pytania o treść raportów.
--         Wzór do podejrzenia: opis sąsiedniej funkcji, który model już dostaje:
--           DESCRIBE FUNCTION EXTENDED workspace.default.get_customer_profile
COMMENT 'TODO'
RETURN (
  SELECT COALESCE(CONCAT(
    'Revenue Summary\n',
    'Segment: ', CASE WHEN segment = -1 THEN 'ALL' ELSE CAST(segment AS STRING) END,
    ' | State: ', state_filter, '\n',
    'Customers: ', CAST(COUNT(*) AS STRING), '\n',
    'Total revenue: $', FORMAT_NUMBER(SUM(monetary), 2), '\n',
    'Avg revenue/customer: $', FORMAT_NUMBER(AVG(monetary), 2), '\n',
    'Total orders: ', CAST(CAST(SUM(num_orders) AS BIGINT) AS STRING), '\n',
    'Customers with orders: ', CAST(SUM(has_orders) AS STRING)
  ),
  -- bez pasujących wierszy SUM daje NULL, a CONCAT z NULL to NULL: model dostałby pusty wynik
  CONCAT('No data for segment ', CAST(segment AS STRING), ' / state ', state_filter, '.'))
  FROM workspace.default.gold_customer_360
  -- TODO 2: filtr, w którym parametr ma wartość "wszystkie".
  --         Wzorzec parametru opcjonalnego w SQL:  (param = <wartość-wszystkie> OR kolumna = param)
  --         Czyta się to tak: albo pytasz o wszystko, albo kolumna ma pasować do parametru.
  --         Potrzebujesz dwóch takich nawiasów połączonych przez AND:
  --           segment       = -1     -> wszystkie segmenty  (kolumna: loyalty_segment)
  --           state_filter  = 'ALL'  -> wszystkie stany     (kolumna: state)
  WHERE TODO
);

-- Test bez modelu: przychód od klientów VIP w NY
SELECT workspace.default.get_revenue_summary(3, 'NY') AS revenue_report;


## 2. Tool calling "ręcznie": cztery kroki

> **Cel:** zobaczyć cztery kroki tool callingu bez frameworka.
> **Gotowe, gdy:** potrafisz wskazać moment, w którym to model, a nie Twój kod, decyduje o wywołaniu.


```
Użytkownik: "Jaki jest łączny przychód od klientów VIP w stanie Nowy Jork?"
  1. model dostaje pytanie + opis narzędzia (JSON Schema)
  2. model wybiera: get_revenue_summary(segment=3, state_filter="NY")
  3. MY wykonujemy funkcję w Unity Catalog (parametry przez args, bez sklejania SQL)
  4. model dostaje wynik i formatuje odpowiedź po polsku
```

W M5 te cztery kroki wykona za nas agent złożony przez `create_agent`. Tu robisz je sam, żeby zobaczyć, co framework robi za Ciebie.

**Lab (opcjonalny):** opisz narzędzie w formacie `tools` (JSON Schema). Nazwy parametrów muszą się zgadzać z funkcją SQL.

In [ ]:
# ZADANIE 4: opis narzędzia dla modelu (JSON Schema).
tools = [{
    "type": "function",
    "function": {
        "name": "get_revenue_summary",
        # TODO: description: co zwraca i kiedy użyć (sens ten sam co COMMENT z zadania 3)
        "description": ...,
        "parameters": {
            "type": "object",
            "properties": {
                # TODO: dwa parametry o nazwach IDENTYCZNYCH jak w funkcji SQL:
                #   segment: integer, opis 0=new, 1=occasional, 2=regular, 3=VIP, -1=all
                #   state_filter: string, opis kod stanu USA albo ALL
            },
            "required": ["segment", "state_filter"],
        },
    },
}]

description = tools[0]["function"]["description"]
properties = tools[0]["function"]["parameters"]["properties"]
if description is Ellipsis or not properties:
    raise NotImplementedError(
        "ZADANIE 4 nie jest uzupełnione: brakuje opisu narzędzia albo parametrów. "
        "Ta komórka przypisuje samą strukturę, więc przeszłaby nawet pusta, "
        "a błąd zobaczyłbyś dopiero przy wysyłaniu narzędzia do modelu."
    )
print(f"Narzędzie dla modelu: {tools[0]['function']['name']}, parametry: {list(properties)}")


**Cztery kroki po kolei.** Opis narzędzia masz wyżej. Niżej widać, co się z nim dzieje: model wybiera funkcję i parametry (kroki 1 i 2), Twój kod ją wykonuje (krok 3), model formatuje odpowiedź z wyniku (krok 4). Krok 3 to zwykły SQL.

> Sprawdź, czy model sam zamienił "VIP" na `3`, a "Nowy Jork" na `NY`. Jeśli tak, zrobił to wyłącznie na podstawie opisu parametrów.


In [ ]:
question = "Jaki jest łączny przychód od klientów VIP w stanie Nowy Jork?"
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]

# Krok 1-2: model wybiera narzędzie i jego parametry
first = llm.chat.completions.create(
    model=LLM_ENDPOINT, messages=messages, tools=tools, tool_choice="auto"
)
message = first.choices[0].message
assert message.tool_calls, f"Model nie wywołał narzędzia: {message.content}"

call = message.tool_calls[0]
arguments = json.loads(call.function.arguments)
print(f"Model wybrał: {call.function.name}({arguments})")

# Krok 3: wykonanie funkcji UC. Parametry idą przez args=, więc nic się z nich nie wykona jako kod.
parameters = {
    "segment": int(arguments.get("segment", -1)),
    "state_filter": str(arguments.get("state_filter", "ALL")),
}
query = f"SELECT {REVENUE_FUNCTION}(:segment, :state_filter)"
result = spark.sql(query, args=parameters).first()[0] or "Brak danych"
print(f"\nWynik funkcji:\n{result}")

# Krok 4: model formatuje odpowiedź z wyniku narzędzia
messages.append(message)
messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
final = llm.chat.completions.create(model=LLM_ENDPOINT, messages=messages)
print(f"\nOdpowiedź:\n{final.choices[0].message.content}")


## 3. Trzy narzędzia agenta

> **Cel:** komplet narzędzi, z których agent w M5 będzie wybierał.
> **Gotowe, gdy:** trzy funkcje są w katalogu i każda zdaje test payloadem.


Agent z M5 dostanie dokładnie te trzy funkcje. Czwarte narzędzie, wyszukiwanie w raportach, dołożymy w M3.

| Funkcja | Język | Odpowiada na | Celowo pomija |
|---|---|---|---|
| `get_average_customer_value(segment)` | SQL | "Jaka jest średnia wartość klienta VIP?" | nic |
| `get_customer_profile(requested_customer_id)` | SQL | "Pokaż profil klienta 123" | `tax_id`, `customer_name`, `lat`, `lon` |
| `format_customer_for_agent(...)` | Python | zamienia metryki klienta na zwięzły tekst | dostęp do tabel (Python UDF ich nie czyta) |

> Funkcja profilu nie zwraca PII. To zabezpieczenie wbudowane w narzędzie, niezależne od promptu. Nawet udany jailbreak nie wyciągnie z niej `tax_id`, bo go tam nie ma.

Dwie komórki SQL niżej zakładają pierwsze dwie funkcje. Obie uwzględniają to, że 143 klientów ma w tabeli po dwa wiersze. `get_average_customer_value` liczy średnią po unikalnych klientach (`DISTINCT`), a `get_customer_profile` zawsze wybiera ten sam z dwóch wierszy (`ORDER BY ... LIMIT 1`). Dzięki temu to samo pytanie daje za każdym razem ten sam wynik. Trzecia funkcja, w Pythonie, jest niżej.


In [ ]:
%sql
CREATE OR REPLACE FUNCTION workspace.default.get_average_customer_value(
  segment BIGINT COMMENT 'Loyalty segment ID (0=new, 1=occasional, 2=regular, 3=VIP). Pass -1 for all segments.'
)
RETURNS DOUBLE
COMMENT 'Returns the current average monetary value (total spend, USD) of customers in the given loyalty segment, computed live from gold_customer_360. Use for numeric questions about customer value. Pass -1 for the overall average.'
-- DISTINCT: 143 klientów ma w tabeli zdublowany wiersz, więc średnia po wierszach lekko kłamie.
-- Narzędzie agenta musi mieć jedną, jawną definicję metryki.
RETURN SELECT ROUND(AVG(monetary), 2)
FROM (
  SELECT DISTINCT customer_id, monetary, loyalty_segment
  FROM workspace.default.gold_customer_360
)
WHERE (segment = -1 OR loyalty_segment = segment);

In [ ]:
%sql
CREATE OR REPLACE FUNCTION workspace.default.get_customer_profile(
  requested_customer_id BIGINT COMMENT 'The numeric customer ID to retrieve.'
)
RETURNS STRING
COMMENT 'Returns an agent-readable B2B customer profile by customer ID: location, loyalty segment, RFM metrics and order history. Never returns PII (no tax_id, customer name or coordinates). Returns NULL when the customer does not exist.'
RETURN SELECT CONCAT_WS(
  '\n',
  CONCAT('Customer ID: ', CAST(customer_id AS STRING)),
  CONCAT('Location: ', COALESCE(city, 'N/A'), ', ', COALESCE(state, 'N/A')),
  CONCAT('Loyalty segment: ', CASE loyalty_segment WHEN 0 THEN 'New (0)' WHEN 1 THEN 'Occasional (1)' WHEN 2 THEN 'Regular (2)' WHEN 3 THEN 'VIP (3)' ELSE 'Unknown' END),
  CONCAT('Units purchased: ', CAST(units_purchased AS STRING)),
  CONCAT('Total spend: $', CAST(ROUND(monetary, 2) AS STRING)),
  CONCAT('Avg item value: $', CAST(ROUND(avg_item_value, 2) AS STRING)),
  CONCAT('Orders: ', CAST(num_orders AS STRING), ' (promo: ', CAST(promo_orders AS STRING), ', ', CAST(ROUND(promo_ratio * 100, 1) AS STRING), '%)'),
  CONCAT('RFM Recency: ', CAST(recency_days AS STRING), ' days'),
  CONCAT('RFM Frequency: ', CAST(frequency AS STRING)),
  CONCAT('Customer since: ', COALESCE(CAST(first_order_date AS STRING), 'no orders')),
  CONCAT('Last order: ', COALESCE(CAST(last_order_date AS STRING), 'no orders'))
)
FROM workspace.default.gold_customer_360
WHERE customer_id = requested_customer_id
-- 143 klientów ma zdublowany wiersz; ORDER BY + LIMIT 1 daje zawsze ten sam wynik,
-- a agent, który dwa razy pyta o tego samego klienta, dostaje dwa razy tę samą odpowiedź.
ORDER BY last_order_date DESC NULLS LAST, monetary DESC
LIMIT 1;

### Funkcja Python w Unity Catalog

> **Cel:** to samo dla funkcji w Pythonie, gdzie kontraktem jest docstring.
> **Gotowe, gdy:** funkcja Python leży w katalogu obok funkcji SQL.


`DatabricksFunctionClient.create_python_function` rejestruje zwykłą funkcję Pythona jako funkcję UC. Docstring staje się jej opisem: linia streszczenia to `COMMENT` funkcji, a sekcja `Args` to opisy parametrów. Model zobaczy dokładnie ten tekst.

Ten kod jest gotowy, bo z `format_customer_for_agent` korzystają M5 i M6. Przeczytaj docstring (streszczenie, `Args:`, `Returns:`). Bez sekcji `Args` funkcja też by się zarejestrowała, ale z ostrzeżeniem, a model dostałby 11 parametrów bez żadnego opisu.

In [ ]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient


def format_customer_for_agent(
    customer_id: int,
    state: str,
    city: str,
    loyalty_segment: int,
    units_purchased: int,
    monetary: float,
    avg_item_value: float,
    num_orders: int,
    promo_orders: int,
    recency_days: int,
    frequency: int,
) -> str:
    """Format one B2B customer's metrics into concise, factual text for an AI agent.

    Args:
        customer_id: Numeric identifier of the customer.
        state: US state of the customer.
        city: City of the customer.
        loyalty_segment: Loyalty tier (0=New, 1=Occasional, 2=Regular, 3=VIP).
        units_purchased: Total units purchased.
        monetary: Total spend in USD (RFM monetary value).
        avg_item_value: Average item value across orders in USD.
        num_orders: Total number of orders.
        promo_orders: Number of promotional order lines.
        recency_days: Days since the last order (RFM recency; 999 means no orders).
        frequency: Number of ordered line items (RFM frequency).

    Returns:
        A newline-separated customer summary without PII, suitable as agent context.
    """
    segment_labels = {0: "New", 1: "Occasional", 2: "Regular", 3: "VIP"}
    return "\n".join([
        f"Customer ID: {customer_id}",
        f"Location: {city}, {state}",
        f"Loyalty segment: {segment_labels.get(loyalty_segment, 'Unknown')} ({loyalty_segment})",
        f"Units purchased: {units_purchased}",
        f"Total spend: ${monetary:,.2f}",
        f"Average item value: ${avg_item_value:,.2f}",
        f"Orders: {num_orders} (promo lines: {promo_orders})",
        f"Recency: {recency_days} days since last order",
        f"Frequency: {frequency}",
    ])

**Rejestracja w katalogu.** Trzy linijki: klient, rejestracja, potwierdzenie. Od tej chwili funkcja Pythona leży w Unity Catalog obok funkcji SQL i agent widzi ją tak samo jak je.


In [ ]:
function_client = DatabricksFunctionClient(execution_mode="serverless")
function_client.create_python_function(
    func=format_customer_for_agent, catalog=CATALOG, schema=SCHEMA, replace=True
)
print(f"Zarejestrowano: {FORMAT_FUNCTION}")


## 4. Test surowym payloadem: unit test narzędzia

> **Cel:** sprawdzić narzędzie bez modelu, bo błąd narzędzia i błąd modelu leczy się inaczej.
> **Gotowe, gdy:** każda funkcja zwraca sensowny wynik i **żadna nie zwraca `tax_id`**.


Zanim oddamy funkcje agentowi, wywołujemy je bez modelu. Wynik poniżej to dokładnie ten tekst albo liczba, które dostanie agent. Jeśli wynik jest nieczytelny dla Ciebie, będzie nieczytelny dla modelu.

Trzy komórki niżej:

- Pierwsza przygotowuje listę `PAYLOADS`: którą funkcję wywołać i z jakimi parametrami. Funkcja w Pythonie ma jedenaście parametrów, więc komórka bierze ich wartości z wiersza klienta X.
- Druga wywołuje każdą funkcję przez `execute_function`, tą samą drogą, którą pójdzie agent, i wypisuje wynik. Jeśli w wyniku pojawi się numer w formacie `tax_id`, komórka zatrzyma się z błędem.
- Trzecia (opcjonalna) pokazuje, jak funkcję widzi model: opis i parametry prosto z Unity Catalog.


In [ ]:
from pyspark.sql import functions as F

# Ten sam porządek co w get_customer_profile: klient ze zdublowanym wierszem daje stały wynik.
customer = (spark.table(GOLD_TABLE)
            .where(f"customer_id = {VIP_CUSTOMER_ID}")
            .orderBy(F.col("last_order_date").desc_nulls_last(), F.col("monetary").desc())
            .first())

# Funkcja Pythona w Unity Catalog ma jedenaście osobnych parametrów, więc wypisujemy je jawnie.
format_arguments = {
    "customer_id": VIP_CUSTOMER_ID,
    "state": customer["state"] or "N/A",
    "city": customer["city"] or "N/A",
    "loyalty_segment": int(customer["loyalty_segment"]),
    "units_purchased": int(customer["units_purchased"] or 0),
    "monetary": float(customer["monetary"]),
    "avg_item_value": float(customer["avg_item_value"]),
    "num_orders": int(customer["num_orders"]),
    "promo_orders": int(customer["promo_orders"]),
    "recency_days": int(customer["recency_days"]),
    "frequency": int(customer["frequency"]),
}

PAYLOADS = [
    ("Średnia wartość: VIP (3)", AVG_VALUE_FUNCTION, {"segment": 3}),
    ("Średnia wartość: wszyscy (-1)", AVG_VALUE_FUNCTION, {"segment": -1}),
    (f"Profil klienta {VIP_CUSTOMER_ID}", PROFILE_FUNCTION,
     {"requested_customer_id": VIP_CUSTOMER_ID}),
    ("Formatowanie (Python)", FORMAT_FUNCTION, format_arguments),
]
print(f"{len(PAYLOADS)} wywołania do sprawdzenia bez modelu.")


In [ ]:
import re

TAX_ID_PATTERN = re.compile(r"\d{2}-\d{7}")  # format tax_id w tych danych: 12-3456789

for label, function_name, parameters in PAYLOADS:
    value = function_client.execute_function(
        function_name=function_name, parameters=parameters
    ).value
    print(f"\n{label}\n{value}")
    assert not TAX_ID_PATTERN.search(str(value)), f"PII (tax_id) w wyniku {function_name}!"

print("\nŻadna funkcja nie zwraca tax_id.")
print("Oczekiwana średnia VIP: 1043.15 (po deduplikacji; po wierszach byłoby 1038.72).")
print(f"Zapamiętaj numer klienta X: {VIP_CUSTOMER_ID}. Użyjesz go w Playground i w M5.")


In [ ]:
# Co widzi model: opis funkcji i parametrów prosto z Unity Catalog
display(spark.sql(f"DESCRIBE FUNCTION EXTENDED {PROFILE_FUNCTION}"))

## Zanim napiszesz funkcję: Databricks ma gotowe funkcje AI

> **Cel:** wiedzieć, czego **nie** trzeba pisać samemu.
> **Gotowe, gdy:** dla swojego zadania potrafisz powiedzieć, czy wystarczy gotowa funkcja AI, czy trzeba własną funkcję UC.

Funkcje AI to wbudowane funkcje SQL, które wołają model za Ciebie. Nie potrzebujesz endpointu, klienta ani Pythona, wystarczy `SELECT`. Z jednej z nich, `ai_parse_document`, korzystamy w M3.

| Funkcja | Co robi | Gdzie się przydaje u nas |
|---|---|---|
| `ai_query` | dowolny prompt na kolumnie, wskazany model | masowe przetwarzanie tekstu wsadem |
| `ai_analyze_sentiment` | wydźwięk tekstu | opinie klientów Bakehouse (ścieżka B) |
| `ai_classify` | przypisanie do jednej z podanych kategorii | routing zgłoszeń, tagowanie opinii |
| `ai_extract` | wyciąga wskazane pola z tekstu | dane z maili i formularzy |
| `ai_mask` | zasłania wskazane typy danych osobowych | czwarta warstwa obrony obok maski z M4 |
| `ai_summarize`, `ai_translate` | streszczenie, tłumaczenie | raporty dla zarządu |
| `ai_parse_document` | PDF na tekst, tabele i opisy wykresów | **M3**, cały RAG stoi na tej funkcji |
| `ai_forecast`, `vector_search` | prognoza szeregu, wyszukiwanie w indeksie | planowanie, RAG w czystym SQL |

**Kiedy gotowa funkcja AI, a kiedy własna funkcja UC:**

| | Funkcja AI | Własna funkcja UC (to, co robimy w M2) |
|---|---|---|
| Co liczy | model, więc wynik bywa różny | SQL, więc wynik jest deterministyczny |
| Kto ustala kształt odpowiedzi | model | Ty, w `RETURN` |
| Do czego | tekst: wydźwięk, kategoria, streszczenie | liczby i fakty z tabeli |
| W agencie | raczej w potoku danych przed agentem | jako narzędzie agenta |

Zasada z M2 zostaje w mocy: liczby bierzemy z funkcji deterministycznej. Funkcja AI świetnie nadaje się do przygotowania danych, a nie do policzenia przychodu.


In [ ]:
%sql
-- Jedno wywołanie gotowej funkcji AI, bez endpointu i bez Pythona: sam SELECT.
-- ai_mask zasłania wskazane typy danych osobowych. To ta sama myśl co maska z M4,
-- tylko liczona modelem, a nie polityką katalogu.
SELECT ai_mask(
  'Kontakt do klienta: Jan Kowalski, jan.kowalski@techretail.example, tel. 600 100 200',
  array('person', 'email', 'phone')
) AS zamaskowane


## 5. Lab w AI Playground: funkcje jako Tools

> **Cel:** te same funkcje jako Tools, bez pisania kodu.
> **Gotowe, gdy:** model sam wybiera funkcję i pokazuje jej wynik.


1. Otwórz **Playground**, wybierz `databricks-meta-llama-3-3-70b-instruct` i wklej `SYSTEM_PROMPT` (wypisany w M1).
2. **Tools → Add tool → Unity Catalog function**. Dodaj `workspace.default.get_average_customer_value` i `workspace.default.get_customer_profile`.
3. Zadaj cztery pytania. Po każdym rozwiń panel narzędzia: zobaczysz wywołanie z parametrami i surowy wynik, identyczny z testem payloadem.

| Pytanie | Oczekiwanie | Co się stało? |
|---|---|---|
| "Jaka jest średnia wartość klienta VIP?" | `get_average_customer_value(segment=3)`, wynik 1043,15 | |
| "Pokaż profil klienta X" (numer z komórki wyżej) | `get_customer_profile(X)` | |
| "Podaj tax_id klienta X." | odmowa; funkcja i tak nie zwraca `tax_id` | |
| "Jaka była sprzedaż w Kanadzie?" | brak narzędzia, uczciwe "nie mam takich danych" | |

4. Porównaj z M1: to samo pytanie o VIP-ów bez narzędzi i z narzędziami.

> **Po co Playground, skoro mamy kod?** To najszybszy test, czy **opis funkcji wystarcza**, żeby model po nią sięgnął. Gdy nie sięga, popraw `COMMENT` (`CREATE OR REPLACE FUNCTION`) i zapytaj jeszcze raz.

## Fallback: co robi dobry agent, gdy nic nie pasuje

> **Cel:** wiedzieć, jak wygląda uczciwa odmowa.
> **Gotowe, gdy:** na pytanie spoza danych agent mówi, że nie ma danych, zamiast zgadywać.


| Sytuacja | Słaby agent | Dobry agent |
|---|---|---|
| Żadne narzędzie nie pasuje | zgaduje odpowiedź z pamięci | "Nie mam takich danych" i propozycja pytania, na które odpowie |
| Narzędzie zwraca pusty wynik (np. nieistniejący klient) | wymyśla liczby | mówi wprost: brak wyników dla tych parametrów |
| Pytanie o PII | odpowiada, bo prompt nie przewidział tej formy | odmawia i proponuje wersję bez PII |
| Błąd wykonania narzędzia | pokazuje stack trace albo milczy | jedna ponowna próba, potem czytelny komunikat |
| Pytanie poza domeną | odpowiada o wszystkim | odmawia i wraca do domeny TechRetail |

Fallback agenta to zdanie w system promptcie (ostatnie dwa zdania `SYSTEM_PROMPT`) plus test, który sprawdza, że działa. Test dopiszemy w M5.

> Uwaga na słowo: w Unity Gateway "fallback" oznacza przełączenie na zapasowy model, gdy endpoint zwraca błąd. To inna rzecz, wrócimy do niej w M6.

## B · Samodzielnie: dwie funkcje i wybór narzędzia (Bakehouse)

> **Cel:** dwie funkcje-narzędzia sieci piekarni Bakehouse w schemacie `workspace.bakehouse`: `bh_payment_methods` (jak płacą klienci franczyzy) i `bh_franchise_summary` (sprzedaż franczyzy).
> **Lekcja:** model wybiera narzędzie wyłącznie po opisie (`COMMENT`), więc dwa narzędzia o jednej franczyzie muszą się różnić opisem, nie samą nazwą.
> **Gotowe, gdy:** obie funkcje zwracają tekst dla pierwszej franczyzy z danych i zdanie "No transactions for franchise -1..." dla nieistniejącej, a w teście wyboru model trafia 2 z 2.

Obie funkcje przyjmują ten sam parametr: numer franczyzy. Model rozróżni je tylko wtedy, gdy każdy `COMMENT` mówi, kiedy funkcji użyć i kiedy nie: płatności to nie przychód. Napisz też, co funkcja zwraca przy braku danych, bo pusty wynik bez komentarza model zastąpi zgadywaniem.

Pierwsza komórka tworzy funkcje i testuje je samym SQL. Druga wysyła modelowi oba narzędzia tym samym mechanizmem co w części 2, tylko opis bierze prosto z katalogu. Kiedy zadziała, zepsuj go celowo: daj obu funkcjom ten sam `COMMENT` ("Returns data about a franchise.") i zobacz, co model wybierze.

In [ ]:
%sql
-- ZADANIE B (1 z 2): narzędzie "jak płacą klienci tej franczyzy".
CREATE OR REPLACE FUNCTION workspace.bakehouse.bh_payment_methods(
  requested_franchise_id BIGINT COMMENT 'Numeric franchise ID from the Bakehouse dataset.'
)
RETURNS STRING
-- TODO: COMMENT: co zwraca, kiedy użyć, kiedy NIE (przychód, produkty),
--       czego nie zwraca (numery kart) i co zwraca, gdy franczyza nie ma transakcji.
COMMENT 'TODO'
-- TODO: liczba transakcji per paymentMethod dla jednej franczyzy, sklejona w jeden tekst.
--       Podpowiedź: podzapytanie z GROUP BY paymentMethod na workspace.bakehouse.transactions,
--       potem concat_ws(', ', collect_list(...)). Uwaga: concat_ws na pustej liście daje '',
--       a nie NULL, więc: COALESCE(NULLIF(..., ''), 'No transactions for franchise ...').
RETURN SELECT 'TODO';
-- Utknąłeś? Rozwiązanie: ../demo/m2_tool_calling, komórka m2-path-b.


In [ ]:
%sql
-- ZADANIE B (2 z 2): narzędzie "sprzedaż i przychód tej franczyzy".
CREATE OR REPLACE FUNCTION workspace.bakehouse.bh_franchise_summary(
  requested_franchise_id BIGINT COMMENT 'Numeric franchise ID from the Bakehouse dataset.'
)
RETURNS STRING
-- TODO: COMMENT, który odróżnia tę funkcję od bh_payment_methods: sprzedaż i przychód franczyzy, nie płatności.
COMMENT 'TODO'
RETURN SELECT CASE
  WHEN COUNT(t.transactionID) = 0
    THEN CONCAT('No transactions for franchise ', CAST(requested_franchise_id AS STRING), ' in the Bakehouse data.')
  ELSE CONCAT_WS(
    '\n',
    CONCAT('Franchise ID: ', CAST(requested_franchise_id AS STRING)),
    CONCAT('Name: ', COALESCE(MAX(f.name), 'N/A'), ' | City: ', COALESCE(MAX(f.city), 'N/A')),
    CONCAT('Transactions: ', CAST(COUNT(t.transactionID) AS STRING)),
    CONCAT('Units sold: ', CAST(SUM(t.quantity) AS STRING)),
    CONCAT('Revenue (USD): ', FORMAT_NUMBER(SUM(t.totalPrice), 2))
  )
END
FROM workspace.bakehouse.transactions t
LEFT JOIN workspace.bakehouse.franchises f ON f.franchiseID = t.franchiseID
WHERE t.franchiseID = requested_franchise_id;
-- Utknąłeś? Rozwiązanie: ../demo/m2_tool_calling, komórka m2-path-b-franchise.


In [ ]:
# Test bez modelu: raz franczyza, która istnieje, raz numer, którego nie ma.
PAYMENT_FUNCTION = f"{CATALOG}.{BH_SCHEMA}.bh_payment_methods"
FRANCHISE_FUNCTION = f"{CATALOG}.{BH_SCHEMA}.bh_franchise_summary"
BH_FID = int(spark.table(BH_TRANSACTIONS).select(F.min("franchiseID")).first()[0])
MISSING_FID = -1


def call_function(function_name: str, franchise_id: int) -> str:
    """args= zamiast sklejania SQL: parametr idzie osobno i nie wykona się jako kod."""
    return spark.sql(f"SELECT {function_name}(:fid)", args={"fid": franchise_id}).first()[0]


for function_name in (PAYMENT_FUNCTION, FRANCHISE_FUNCTION):
    found = call_function(function_name, BH_FID)
    missing = call_function(function_name, MISSING_FID)
    print(f"\n{function_name}({BH_FID})\n{found}")
    print(f"{function_name}({MISSING_FID})\n{missing}")
    assert found and not found.startswith("No transactions"), \
        f"{function_name}: brak wyniku dla franczyzy {BH_FID}"
    assert missing and missing.startswith("No transactions"), \
        f"{function_name}: pusty wynik bez komunikatu: {missing!r}"

print(f"\nObie funkcje odpowiadają dla franczyzy {BH_FID} i mówią wprost, gdy franczyzy nie ma.")


**Który opis wybierze model?** Dwa pytania o tę samą franczyzę, oba narzędzia naraz. Opis każdego narzędzia to jego `COMMENT` odczytany z `information_schema.routines`, więc model widzi dokładnie to, co napisałeś w `CREATE FUNCTION`.

In [ ]:
# Opis narzędzia bierzemy prosto z katalogu: model zobaczy dokładnie ten tekst,
# który wpisałeś w COMMENT funkcji.
BH_FUNCTIONS = ["bh_payment_methods", "bh_franchise_summary"]

routines = spark.sql(
    f"SELECT routine_name, comment FROM {CATALOG}.information_schema.routines "
    f"WHERE routine_schema = '{BH_SCHEMA}'"
).collect()
comments = {row["routine_name"]: row["comment"] for row in routines
            if row["routine_name"] in BH_FUNCTIONS}
missing = sorted(set(BH_FUNCTIONS) - set(comments))
assert not missing, f"Brak funkcji w {CATALOG}.{BH_SCHEMA}: {missing}"

FRANCHISE_ID_PARAMETER = {"type": "integer",
                          "description": "Numeric franchise ID from the Bakehouse dataset."}
bh_tools = [{
    "type": "function",
    "function": {
        "name": name,
        "description": comments[name],
        "parameters": {
            "type": "object",
            "properties": {"requested_franchise_id": FRANCHISE_ID_PARAMETER},
            "required": ["requested_franchise_id"],
        },
    },
} for name in BH_FUNCTIONS]
print(f"Narzędzia dla modelu: {BH_FUNCTIONS}")


In [ ]:
# ZADANIE B: drugie pytanie i narzędzie, które powinno na nie odpowiedzieć.
BH_SYSTEM_PROMPT = ("Jesteś asystentem sieci piekarni Bakehouse. Odpowiadaj po polsku. "
                    "Liczby bierz wyłącznie z narzędzi.")

cases = [
    (f"Jak płacą klienci franczyzy {BH_FID}: gotówką czy kartą?", "bh_payment_methods"),
    # TODO: pytanie o tę samą franczyzę, na które ma odpowiedzieć bh_franchise_summary,
    #       i nazwa tej funkcji, np. (f"Jaki przychód ... {BH_FID}?", "bh_franchise_summary")
    (..., ...),
]
if any(item is Ellipsis for case in cases for item in case):
    raise NotImplementedError("Uzupełnij TODO: drugie pytanie i oczekiwane narzędzie w `cases`.")

hits = 0
for question, expected in cases:
    reply = llm.chat.completions.create(
        model=LLM_ENDPOINT,
        messages=[{"role": "system", "content": BH_SYSTEM_PROMPT},
                  {"role": "user", "content": question}],
        tools=bh_tools,
        tool_choice="auto",
        temperature=0,
        max_tokens=200,
    )
    calls = reply.choices[0].message.tool_calls or []
    chosen = calls[0].function.name if calls else "(brak wywołania)"
    arguments = calls[0].function.arguments if calls else ""
    hits += int(chosen == expected)
    mark = "OK" if chosen == expected else "BŁĄD"
    print(f"{mark} {question}\n   model wybrał: {chosen}({arguments}), oczekiwane: {expected}")

print(f"\nTrafienia: {hits}/{len(cases)}.")
if hits < len(cases):
    print("Model pomylił narzędzia: dopisz w COMMENT, kiedy funkcji NIE używać, "
          "i uruchom obie komórki jeszcze raz.")


## C · Wyzwanie: funkcja, która mówi "nie mam danych", i grant, który znika (Airbnb)

> **Cel:** funkcja `workspace.airbnb.get_airbnb_summary(neighbourhood)` na tabeli `workspace.airbnb.listings`: liczba ofert, w tym krótkoterminowych (`is_short_term`), mediana ceny z poprawnych cen (`price_valid`) i najczęstszy typ pokoju. Do tego `GRANT EXECUTE` dla grupy.
> **Lekcja:** narzędzie agenta musi odpowiadać zdaniem także wtedy, gdy danych nie ma, a każda poprawka przez `CREATE OR REPLACE FUNCTION` kasuje jego granty.
> **Gotowe, gdy:** `get_airbnb_summary('Mission')` zwraca 789 ofert, zmyślona dzielnica zwraca "Brak ofert w dzielnicy ...", a po `CREATE OR REPLACE` `SHOW GRANTS` nie pokazuje dla Twojej grupy żadnego grantu nadanego na samej funkcji, a po ponownym `GRANT` pokazuje jeden.

Pusty wynik trzeba obsłużyć. Funkcja, która przy braku danych zwraca `NULL`, każe modelowi zgadywać, a model zgadnie. Funkcja, która zwraca zdanie "brak ofert w dzielnicy X", daje mu treść do zacytowania.

Dwie decyzje, które najczęściej wychodzą źle:

- **Pusty wynik.** Agregat na zerze wierszy zwraca jeden wiersz z `NULL` w medianie. `CONCAT` z `NULL` daje `NULL`, więc `COALESCE` przechodzi na zdanie "Brak ofert". Nie trzeba `CASE WHEN COUNT(*) = 0`.
- **Grupa.** `GRANT` w Unity Catalog przyjmuje grupy konta. Lokalne grupy workspace'u (`admins`, `users` na trialu) dają błąd. `account users` istnieje zawsze, więc rozwiązanie jej używa. Ty możesz wpisać grupę ze swojego konta (Settings → Identity and access → Groups).

Mediana liczona ze wszystkich cen łapie oferty po 0 USD i powyżej 2000 USD, dlatego liczymy ją tylko z `price_valid`. Liczba ofert krótkoterminowych pokazuje, że 45% ofert w tych danych to najem od 30 nocy.

> **Sprawdzone na tym warsztacie.** Service principal z samym `EXECUTE` na funkcji, bez `SELECT` na tabeli, wywołuje ją poprawnie: funkcja SQL działa z uprawnieniami właściciela. Druga funkcja, bez `EXECUTE`, zwraca `INSUFFICIENT_PERMISSIONS`. To jest cała treść zasady najmniejszych uprawnień dla narzędzi agenta.
>
> **Pułapka przy poprawianiu funkcji:** `CREATE OR REPLACE FUNCTION` **kasuje wszystkie granty**. Po każdej poprawce `COMMENT` trzeba nadać je ponownie. Dlatego ostatni krok poniżej to `SHOW GRANTS`, a nie `GRANT`.

In [ ]:
%sql
-- ZADANIE C: narzędzie dla ofert Airbnb w jednej dzielnicy San Francisco.
CREATE OR REPLACE FUNCTION workspace.airbnb.get_airbnb_summary(
  requested_neighbourhood STRING COMMENT 'San Francisco neighbourhood name, e.g. Mission.'
)
RETURNS STRING
-- TODO: COMMENT: co zwraca, kiedy użyć, kiedy NIE i co robi przy braku danych
COMMENT 'TODO'
RETURN SELECT COALESCE(
  -- TODO: podsumowanie dzielnicy: liczba ofert, liczba krótkoterminowych (is_short_term),
  --       mediana ceny tylko z price_valid, najczęstszy typ pokoju (mode(room_type))
  NULL,
  CONCAT('Brak ofert w dzielnicy ', requested_neighbourhood, '.')
)
FROM workspace.airbnb.listings
WHERE neighbourhood = requested_neighbourhood;
-- Utknąłeś? Rozwiązanie: ../demo/m2_tool_calling, komórka m2-path-c.


In [ ]:
# Test bez modelu: raz dzielnica, która istnieje, raz nazwa zmyślona.
AIRBNB_FUNCTION = f"{CATALOG}.{AIRBNB_SCHEMA}.get_airbnb_summary"
MISSING_NEIGHBOURHOOD = "Nie ma takiej dzielnicy"

checks = spark.sql(
    f"SELECT {AIRBNB_FUNCTION}('Mission') AS istnieje, "
    f"{AIRBNB_FUNCTION}('{MISSING_NEIGHBOURHOOD}') AS pusto"
).first()
print(checks["istnieje"])
print(checks["pusto"])

assert " 789 ofert" in (checks["istnieje"] or ""), \
    f"Mission ma mieć 789 ofert, jest: {checks['istnieje']!r}"
assert checks["pusto"] == f"Brak ofert w dzielnicy {MISSING_NEIGHBOURHOOD}.", \
    f"Pusty wynik bez komunikatu: {checks['pusto']!r}"


In [ ]:
# Nadajemy prawo wykonania funkcji grupie konta i sprawdzamy, kto je ma.
MY_GROUP = "account users"   # grupa konta; lokalne grupy workspace'u dają błąd przy GRANT


def grant_execute(function_name: str, group: str) -> bool:
    """Nadaje EXECUTE grupie. Na Free Edition grupy konta bywają niedostępne: wtedy False."""
    try:
        spark.sql(f"GRANT EXECUTE ON FUNCTION {function_name} TO `{group}`")
        return True
    except Exception as error:
        print(f"GRANT dla `{group}` nie przeszedł ({type(error).__name__}). "
              f"Lekcję zobacz na demo prowadzącego.")
        return False


GRANT_WORKS = grant_execute(AIRBNB_FUNCTION, MY_GROUP)
if GRANT_WORKS:
    display(spark.sql(f"SHOW GRANTS ON FUNCTION {AIRBNB_FUNCTION}"))


In [ ]:
%sql
-- Ta sama funkcja, poprawiony COMMENT i nic poza tym. Uruchom, a potem sprawdź granty w komórce niżej.
CREATE OR REPLACE FUNCTION workspace.airbnb.get_airbnb_summary(
  requested_neighbourhood STRING COMMENT 'San Francisco neighbourhood name, e.g. Mission.'
)
RETURNS STRING
COMMENT 'Podsumowanie ofert Airbnb w jednej dzielnicy San Francisco (liczba ofert i krótkoterminowych, mediana poprawnych cen za noc, najczęstszy typ pokoju). Tylko dla pytań o konkretną dzielnicę; przy braku danych zwraca zdanie Brak ofert w dzielnicy X.'
RETURN SELECT COALESCE(
  CONCAT('Dzielnica ', requested_neighbourhood, ': ', CAST(COUNT(*) AS STRING), ' ofert, w tym ',
         CAST(COUNT_IF(is_short_term) AS STRING), ' krótkoterminowych (poniżej 30 nocy), mediana ceny ',
         CAST(CAST(percentile_approx(CASE WHEN price_valid THEN price END, 0.5) AS BIGINT) AS STRING),
         ' USD za noc, najczęstszy typ: ', mode(room_type), '.'),
  CONCAT('Brak ofert w dzielnicy ', requested_neighbourhood, '.')
)
FROM workspace.airbnb.listings
WHERE neighbourhood = requested_neighbourhood;


In [ ]:
# Wniosek z zadania: CREATE OR REPLACE skasował grant, choć zmieniliśmy tylko opis funkcji.
# SHOW GRANTS pokazuje też uprawnienia odziedziczone z katalogu i schematu, dlatego liczymy
# wyłącznie wiersze nadane na samej funkcji (ObjectType FUNCTION).
if not GRANT_WORKS:
    print("Bez grup konta nie zobaczysz tego na własnym workspace. Popatrz na ekran prowadzącego.")
else:
    grants = spark.sql(f"SHOW GRANTS ON FUNCTION {AIRBNB_FUNCTION}")
    display(grants)
    grants_now = grants.where(f"Principal = '{MY_GROUP}' AND ObjectType = 'FUNCTION'").count()
    print(f"Grantów na funkcji dla {MY_GROUP} po CREATE OR REPLACE: {grants_now} (oczekiwane 0)")
    assert grants_now == 0, "Grant na funkcji przetrwał CREATE OR REPLACE, a nie powinien"

    grant_execute(AIRBNB_FUNCTION, MY_GROUP)
    print("Grant nadany ponownie. Każda poprawka funkcji to ponowne nadanie uprawnień.")


## Karta wzorca: narzędzie tabelaryczne

1. **Jedno pytanie biznesowe = jedna funkcja** (małe, jednoznaczne, deterministyczne).
2. **`COMMENT`:** co zwraca, kiedy użyć, kiedy nie i czego celowo nie zwraca.
3. **Bez danych wrażliwych** w wyniku; SQL czyta dane, Python tylko formatuje.
4. **Test bez modelu** (`execute_function`), potem **Playground**. Gdy model nie sięga po funkcję, popraw `COMMENT`.

**Canvas agenta** (`workshop/transfer/canvas_agenta.md`): dla 2 pytań z listy zapisz nazwę funkcji, parametr i jedno zdanie `COMMENT`.

## Podsumowanie

- Narzędzie to funkcja **opisana dla modelu**: nazwa, opis, parametry, wynik. Model widzi tylko opis, więc przy złym wyborze narzędzia poprawiasz opis.
- Tool calling to cztery kroki: model wybiera, Ty wykonujesz, model formatuje. Agent z `create_agent` w M5 robi tę pętlę za Ciebie.
- Funkcje Unity Catalog to kontrakt między danymi a agentem: stały kształt odpowiedzi, bez PII, testowalny bez modelu, z uprawnieniem `EXECUTE`.
- Test surowym payloadem to unit test narzędzia. Rób go przed podpięciem do agenta.

**Dalej:** M3. Drugie źródło wiedzy: raporty PDF i AI Search.